# InternVL3 Document-Aware Batch Processing (Clean Version)

Simplified batch processing notebook that avoids the infinite recursion issue from BatchDocumentProcessor.

**Features:**
- Direct processing pattern (no complex inheritance)
- Working prompt configuration that restored accuracy
- Simple, maintainable code structure
- Standard analytics and reporting

## 1. Imports

In [ ]:
# Core imports
import os
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rich import print as rprint
from rich.console import Console

warnings.filterwarnings('ignore')
console = Console()

# Set project root
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

# Import only what we need - NO BatchDocumentProcessor!
from common.batch_analytics import BatchAnalytics
from common.batch_reporting import BatchReporter
from common.batch_visualizations import BatchVisualizer
from common.evaluation_metrics import load_ground_truth
from common.extraction_parser import discover_images, parse_extraction_response
from common.unified_schema import DocumentTypeFieldSchema
from models.document_aware_internvl3_processor import DocumentAwareInternVL3Processor

print("✅ Imports completed successfully")
print(f"📁 Project root: {project_root}")

## 2. Configuration

In [2]:
# Configuration - Simple and clean
CONFIG = {
    # Model settings
    'MODEL_PATH': "/home/jovyan/nfs_share/models/InternVL3-2B",  # Jupyter environment path
    
    # Batch settings
    'DATA_DIR': 'evaluation_data',
    'GROUND_TRUTH': 'evaluation_data/ground_truth.csv',
    'MAX_IMAGES': None,  # None for all, or set limit
    'DOCUMENT_TYPES': None,  # None for all, or ['invoice', 'receipt']
    
    # Output settings
    'OUTPUT_BASE': os.getenv('OUTPUT_DIR', 'output'),
    'VERBOSE': True,
    
    # V100 optimization settings
    'USE_QUANTIZATION': True,
    'DEVICE_MAP': 'auto',
    'MAX_NEW_TOKENS': 4000,
    'TORCH_DTYPE': 'bfloat16',
    'LOW_CPU_MEM_USAGE': True
}

# CRITICAL: Keep the working prompt configuration that fixed accuracy!
PROMPT_CONFIG = {
    'detection_file': 'prompts/document_type_detection.yaml',
    'detection_key': 'detection',
    'extraction_files': {
        'INVOICE': 'prompts/internvl3_invoice_extraction.yaml',
        'RECEIPT': 'prompts/internvl3_receipt_extraction.yaml',
        'BANK_STATEMENT': 'prompts/internvl3_bank_statement_extraction.yaml'
    },
    'extraction_keys': {
        'INVOICE': 'standard',
        'RECEIPT': 'standard', 
        'BANK_STATEMENT': 'flat'  # Use flat for bank statements
    }
}

print("✅ Configuration loaded")
print(f"🎯 Model: {CONFIG['MODEL_PATH']}")
print(f"📊 Prompt config: InternVL3-specific prompts with format enforcement")

✅ Configuration loaded
🎯 Model: /home/jovyan/nfs_share/models/InternVL3-2B
📊 Prompt config: InternVL3-specific prompts with format enforcement


## 3. Output Directory Setup

In [3]:
# Setup output directories
OUTPUT_BASE = Path(CONFIG['OUTPUT_BASE'])
if not OUTPUT_BASE.is_absolute():
    OUTPUT_BASE = Path.cwd() / OUTPUT_BASE

BATCH_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_DIRS = {
    'base': OUTPUT_BASE,
    'batch': OUTPUT_BASE / 'batch_results',
    'csv': OUTPUT_BASE / 'csv',
    'visualizations': OUTPUT_BASE / 'visualizations',
    'reports': OUTPUT_BASE / 'reports'
}

for dir_path in OUTPUT_DIRS.values():
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"✅ Output directories created")
print(f"📁 Base: {OUTPUT_BASE}")
print(f"⏰ Timestamp: {BATCH_TIMESTAMP}")

✅ Output directories created
📁 Base: /home/jovyan/nfs_share/tod/LMM_POC/output
⏰ Timestamp: 20250915_030318


## 4. Initialize Schema and Processor

In [ ]:
# Initialize document schema
schema_loader = DocumentTypeFieldSchema(model="internvl3")
print(f"✅ Schema loaded with {schema_loader.total_fields} total fields")

# NOTE: We'll initialize the processor directly in the batch processing loop
# This avoids loading the model until we actually need it
# The DocumentAwareInternVL3Processor will handle its own initialization
print("✅ Schema ready for document-aware processing")
print("🎯 Processor will be initialized during batch processing (lazy loading)")

## 5. Image Discovery

In [ ]:
# Discover and filter images
all_images = discover_images(CONFIG['DATA_DIR'])
ground_truth = load_ground_truth(CONFIG['GROUND_TRUTH'], verbose=CONFIG['VERBOSE'])

# Apply filters
if CONFIG['DOCUMENT_TYPES']:
    filtered = []
    for img in all_images:
        img_name = Path(img).name
        if img_name in ground_truth:
            doc_type = ground_truth[img_name].get('DOCUMENT_TYPE', '').lower()
            if any(dt.lower() in doc_type for dt in CONFIG['DOCUMENT_TYPES']):
                filtered.append(img)
    all_images = filtered

if CONFIG['MAX_IMAGES']:
    all_images = all_images[:CONFIG['MAX_IMAGES']]

rprint(f"[bold green]Ready to process {len(all_images)} images[/bold green]")
for i, img in enumerate(all_images[:5], 1):
    print(f"  {i}. {Path(img).name}")
if len(all_images) > 5:
    print(f"  ... and {len(all_images) - 5} more")

## 6. Direct Batch Processing (No Recursion!)

In [ ]:
# Direct batch processing - Simple loop, no complex processors!
batch_results = []
processing_times = []
document_types_found = {}

rprint("[bold blue]Starting batch processing...[/bold blue]")

# Initialize processor once (lazy loading)
processor = None

for i, image_path in enumerate(all_images, 1):
    image_name = Path(image_path).name
    
    if CONFIG['VERBOSE']:
        rprint(f"\n[cyan]Processing {i}/{len(all_images)}: {image_name}[/cyan]")
    
    # Initialize processor on first use (lazy loading)
    if processor is None:
        rprint("[yellow]Initializing InternVL3 processor (first image)...[/yellow]")
        try:
            processor = DocumentAwareInternVL3Processor(
                model_path=CONFIG['MODEL_PATH'],
                prompt_config=PROMPT_CONFIG,
                schema_loader=schema_loader,
                verbose=False  # Less verbose for cleaner output
            )
            rprint("[green]✅ Processor initialized successfully[/green]")
        except Exception as e:
            rprint(f"[red]❌ Failed to initialize processor: {e}[/red]")
            break
    
    start_time = time.time()
    
    try:
        # Direct processing - avoids infinite recursion!
        result = processor.process_single_image(
            str(image_path),
            prompt_config=PROMPT_CONFIG
        )
        
        processing_time = time.time() - start_time
        
        # Get document type
        doc_type = result.get('document_type', 'unknown').lower()
        
        # Get extracted data
        extracted_data = result.get('extracted_data', {})
        
        # Evaluate against ground truth if available
        evaluation = {}
        if image_name in ground_truth:
            gt_data = ground_truth[image_name]
            
            # Get expected fields for document type
            expected_fields = schema_loader.get_document_fields(doc_type)
            
            # Calculate accuracy
            fields_matched = 0
            fields_extracted = 0
            
            for field in expected_fields:
                if field in extracted_data and extracted_data[field] != 'NOT_FOUND':
                    fields_extracted += 1
                    if field in gt_data:
                        # Simple exact match for now
                        if str(extracted_data[field]).lower() == str(gt_data[field]).lower():
                            fields_matched += 1
            
            overall_accuracy = fields_matched / len(expected_fields) if expected_fields else 0
            
            evaluation = {
                'overall_accuracy': overall_accuracy,
                'fields_extracted': fields_extracted,
                'fields_matched': fields_matched,
                'total_fields': len(expected_fields)
            }
        
        # Store complete result
        batch_result = {
            'image_path': str(image_path),
            'image_name': image_name,
            'document_type': doc_type,
            'extraction_result': {
                'extracted_data': extracted_data,
                'document_type': doc_type
            },
            'evaluation': evaluation,
            'processing_time': processing_time
        }
        
        batch_results.append(batch_result)
        processing_times.append(processing_time)
        
        # Track document types
        document_types_found[doc_type] = document_types_found.get(doc_type, 0) + 1
        
        if CONFIG['VERBOSE']:
            if evaluation:
                acc_pct = evaluation['overall_accuracy'] * 100
                rprint(f"  ✅ Success - Type: {doc_type}, Accuracy: {acc_pct:.1f}%, Time: {processing_time:.2f}s")
            else:
                rprint(f"  ✅ Success - Type: {doc_type}, Time: {processing_time:.2f}s")
                
    except Exception as e:
        rprint(f"  ❌ Error: {str(e)}")
        batch_results.append({
            'image_path': str(image_path),
            'image_name': image_name,
            'error': str(e),
            'processing_time': time.time() - start_time
        })
        processing_times.append(time.time() - start_time)

# Summary
rprint(f"\n[bold green]✅ Processed {len(batch_results)} images[/bold green]")
if processing_times:
    rprint(f"[cyan]Average time: {np.mean(processing_times):.2f}s[/cyan]")
rprint(f"[cyan]Document types found: {document_types_found}[/cyan]")

## 7. Generate Analytics

In [ ]:
# Create analytics using standard BatchAnalytics
analytics = BatchAnalytics(batch_results, processing_times)

# Generate and save DataFrames
saved_files, df_results, df_summary, df_doctype_stats, df_field_stats = analytics.save_all_dataframes(
    OUTPUT_DIRS['csv'], BATCH_TIMESTAMP, verbose=CONFIG['VERBOSE']
)

# Display key results
rprint("\n[bold blue]📊 Results Summary[/bold blue]")
display(df_summary)

if not df_doctype_stats.empty:
    rprint("\n[bold blue]📋 Document Type Performance[/bold blue]")
    display(df_doctype_stats)

## 8. Export Model-Specific CSV

In [ ]:
# Create model-specific CSV file for model comparison
FIELD_COLUMNS = [
    'DOCUMENT_TYPE', 'BUSINESS_ABN', 'SUPPLIER_NAME', 'BUSINESS_ADDRESS', 
    'PAYER_NAME', 'PAYER_ADDRESS', 'INVOICE_DATE', 'LINE_ITEM_DESCRIPTIONS',
    'LINE_ITEM_QUANTITIES', 'LINE_ITEM_PRICES', 'LINE_ITEM_TOTAL_PRICES',
    'IS_GST_INCLUDED', 'GST_AMOUNT', 'TOTAL_AMOUNT', 'STATEMENT_DATE_RANGE',
    'TRANSACTION_DATES', 'TRANSACTION_AMOUNTS_PAID', 'TRANSACTION_AMOUNTS_RECEIVED',
    'ACCOUNT_BALANCE'
]

# Detect model variant from path
model_variant = "8B" if "8B" in CONFIG['MODEL_PATH'] else "2B"
model_suffix = f"internvl3_{model_variant.lower()}"

# Create comprehensive results data
internvl3_csv_data = []

for i, result in enumerate(batch_results):
    image_name = result['image_name']
    doc_type = result.get('document_type', '').lower()
    processing_time = result.get('processing_time', 0)
    
    # Extract fields from result
    extraction_result = result.get('extraction_result', {})
    extracted_fields = extraction_result.get('extracted_data', {})
    evaluation = result.get('evaluation', {})
    
    # Calculate metrics
    overall_accuracy = evaluation.get('overall_accuracy', 0) * 100 if evaluation else 0
    fields_extracted = evaluation.get('fields_extracted', 0) if evaluation else 0
    fields_matched = evaluation.get('fields_matched', 0) if evaluation else 0
    total_fields = evaluation.get('total_fields', len(FIELD_COLUMNS)) if evaluation else len(FIELD_COLUMNS)
    
    # Create row data
    row_data = {
        'image_file': image_name,
        'image_name': image_name,
        'document_type': doc_type,
        'processing_time': processing_time,
        'field_count': total_fields,
        'found_fields': fields_extracted,
        'field_coverage': (fields_extracted / total_fields * 100) if total_fields > 0 else 0,
        'prompt_used': f"{model_suffix}_{doc_type}",
        'timestamp': datetime.now().isoformat(),
        'overall_accuracy': overall_accuracy,
        'fields_extracted': fields_extracted,
        'fields_matched': fields_matched,
        'total_fields': total_fields
    }
    
    # Add all field values
    for field in FIELD_COLUMNS:
        row_data[field] = extracted_fields.get(field, 'NOT_FOUND')
    
    internvl3_csv_data.append(row_data)

# Create DataFrame and save
internvl3_df = pd.DataFrame(internvl3_csv_data)
internvl3_csv_path = OUTPUT_DIRS['csv'] / f"{model_suffix}_batch_results_{BATCH_TIMESTAMP}.csv"
internvl3_df.to_csv(internvl3_csv_path, index=False)

rprint(f"\n[bold green]✅ InternVL3 model-specific CSV exported:[/bold green]")
rprint(f"[cyan]📄 File: {internvl3_csv_path}[/cyan]")
rprint(f"[cyan]📊 Structure: {len(internvl3_df)} rows × {len(internvl3_df.columns)} columns[/cyan]")

# Display sample
if len(internvl3_df) > 0:
    sample_cols = ['image_file', 'document_type', 'overall_accuracy', 'processing_time', 'found_fields']
    rprint("\n[bold blue]📋 Sample exported data:[/bold blue]")
    display(internvl3_df[sample_cols].head(3))

## 9. Create Visualizations

In [ ]:
# Create visualizations using standard BatchVisualizer
visualizer = BatchVisualizer()

viz_files = visualizer.create_all_visualizations(
    df_results, 
    df_doctype_stats,
    OUTPUT_DIRS['visualizations'],
    BATCH_TIMESTAMP,
    show=False  # Don't display inline to keep output clean
)

if viz_files:
    rprint(f"[bold green]✅ Created {len(viz_files)} visualizations[/bold green]")

## 10. Generate Reports

In [ ]:
# Generate reports using standard BatchReporter
reporter = BatchReporter(
    batch_results, 
    processing_times,
    document_types_found,
    BATCH_TIMESTAMP
)

# Save all reports
report_files = reporter.save_all_reports(
    OUTPUT_DIRS,
    df_results,
    df_summary,
    df_doctype_stats,
    CONFIG['MODEL_PATH'],
    {
        'data_dir': CONFIG['DATA_DIR'],
        'ground_truth': CONFIG['GROUND_TRUTH'],
        'max_images': CONFIG['MAX_IMAGES'],
        'document_types': CONFIG['DOCUMENT_TYPES']
    },
    {
        'use_quantization': CONFIG['USE_QUANTIZATION'],
        'device_map': CONFIG['DEVICE_MAP'],
        'max_new_tokens': CONFIG['MAX_NEW_TOKENS'],
        'torch_dtype': CONFIG['TORCH_DTYPE'],
        'low_cpu_mem_usage': CONFIG['LOW_CPU_MEM_USAGE']
    },
    verbose=CONFIG['VERBOSE']
)

if report_files:
    rprint(f"[bold green]✅ Generated {len(report_files)} reports[/bold green]")

## 11. Final Summary

In [ ]:
# Display final summary
console.rule("[bold green]Batch Processing Complete[/bold green]")

total_images = len(batch_results)
successful = len([r for r in batch_results if 'error' not in r])
avg_accuracy = df_results['overall_accuracy'].mean() if len(df_results) > 0 else 0

rprint(f"[bold green]✅ Processed: {total_images} images[/bold green]")
rprint(f"[cyan]Success Rate: {(successful/total_images*100):.1f}%[/cyan]")
rprint(f"[cyan]Average Accuracy: {avg_accuracy:.2f}%[/cyan]")
rprint(f"[cyan]Average Time: {np.mean(processing_times):.2f}s[/cyan]")
rprint(f"[cyan]Output: {OUTPUT_BASE}[/cyan]")

# Display dashboard if available
dashboard_files = list(OUTPUT_DIRS['visualizations'].glob(f"dashboard_{BATCH_TIMESTAMP}.png"))
if dashboard_files:
    from IPython.display import Image, display
    dashboard_path = dashboard_files[0]
    rprint("\n[bold blue]📊 Visual Dashboard:[/bold blue]")
    display(Image(str(dashboard_path)))

rprint("\n[bold green]🎉 Clean InternVL3 batch processing complete![/bold green]")
rprint("[cyan]✨ No infinite recursion, simple direct processing, maintained accuracy![/cyan]")